# Step 2: Clean the data into two tables

**Goal:** split the raw rows into
- **purchases**: real sales (used for RFM, the target and most features)
- **returns**: cancellations, kept for a later return-rate feature

Rules, applied in this order (the logic lives in `src/step2_clean.py`):
1. Drop rows with no Customer ID. We can't track an unknown customer.
2. Invoices starting with `C` are cancellations and go to the returns table.
3. From the rest, drop `Quantity <= 0`.
4. From the rest, drop `Price <= 0`.
5. Add `TotalPrice = Quantity * Price`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from step1_load_explore import load_raw
from step2_clean import clean, PURCHASES, RETURNS

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Run the cleaning and check the row accounting

Every raw row must end up in exactly one place. `clean()` asserts that the pieces add up to the raw total.

In [2]:
raw = load_raw()
purchases, returns, acct = clean(raw)
pd.Series(acct, name="rows").to_frame()

,rows
raw rows,1067371
dropped: no Customer ID,243007
moved to returns (C invoices),18744
dropped: Quantity <= 0,0
dropped: Price <= 0,71
purchases kept,805549


Notice `dropped: Quantity <= 0` is **0**. Every negative quantity was already on a `C` invoice, so rule 2 caught them. The negatives we saw in step 1 were really cancellations.

In [3]:
# Proof: no negative quantity exists outside cancelled invoices
not_cancel = ~raw["Invoice"].astype(str).str.startswith("C")
print("Negative-quantity rows on non-C invoices:", ((raw["Quantity"] <= 0) & not_cancel & raw["Customer ID"].notna()).sum())

Negative-quantity rows on non-C invoices: 0


## 2. The purchases table

In [4]:
print("customers:", purchases["Customer ID"].nunique(), "| invoices:", purchases["Invoice"].nunique())
print("dates:", purchases["InvoiceDate"].min(), "->", purchases["InvoiceDate"].max())
purchases.head()

customers: 5878 | invoices: 36969
dates: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [5]:
purchases[["Quantity", "Price", "TotalPrice"]].describe()

,Quantity,Price,TotalPrice
count,805549.000000,805549.000000,805549.000000
mean,13.290522,3.206561,22.026505
std,143.634088,29.199173,224.041928
min,1.000000,0.001000,0.001000
25%,2.000000,1.250000,4.950000
50%,5.000000,1.950000,11.850000
75%,12.000000,3.750000,19.500000
max,80995.000000,10953.500000,168469.600000


## 3. The returns table

`TotalPrice` is negative here (money handed back). Some returns belong to customers who have no purchases in our data; those will simply be ignored when we build per-customer features.

In [6]:
print("customers:", returns["Customer ID"].nunique(), "| invoices:", returns["Invoice"].nunique())
print("return rows whose customer never purchased:", (~returns["Customer ID"].isin(purchases["Customer ID"])).sum())
print("customers with both purchases and returns:", len(set(returns["Customer ID"]) & set(purchases["Customer ID"])))
returns.head()

customers: 2572 | invoices: 7901
return rows whose customer never purchased: 129
customers with both purchases and returns: 2511


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4


## 4. Open decision A: non-product stock codes

Real products have codes starting with 5 digits. Anything else is a fee or an internal entry, yet it is still counted as customer spend.

In [7]:
odd = purchases[~purchases["StockCode"].astype(str).str.match(r"^\d{5}")]
(odd.groupby("StockCode")["TotalPrice"].agg(["count", "sum"])
    .sort_values("sum", ascending=False).head(12))

,count,sum
StockCode,,
M,709,152340.570
POST,1838,126563.040
C2,253,12796.000
DOT,16,11906.360
ADJUST,32,3538.520
ADJUST2,3,731.050
BANK CHARGES,32,465.001
D,5,397.890
TEST001,9,225.000


- `POST`, `C2`, `DOT`: postage / carriage. Arguably real customer spend.
- `M` (Manual), `ADJUST`, `ADJUST2`, `BANK CHARGES`, `TEST001`, `TEST002`, `D`, `SP1002`, `PADS`: internal entries, not customer behaviour.

**Suggestion (not applied yet):** remove only the second group, keep postage. Waiting for your decision.

## 5. Open decision B: extreme orders

A few single lines are huge. One of them could dominate a customer's 6-month spend, and so dominate the model.

In [8]:
cols = ["Invoice", "StockCode", "Description", "Quantity", "Price", "TotalPrice", "Customer ID"]
purchases.nlargest(8, "TotalPrice")[cols]

,Invoice,StockCode,Description,Quantity,Price,TotalPrice,Customer ID
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6,16446
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.6,12346
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.0,15098
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.4,15838
135013,502263,M,Manual,1,10953.50,10953.5,12918
358639,524159,M,Manual,1,10468.80,10468.8,14063
74356,496115,M,Manual,1,8985.60,8985.6,17949
228042,511465,15044A,PINK PAPER PARASOL,3500,2.55,8925.0,18008


Does a customer with a giant order also have a matching cancellation? Check the returns table for the top customers:

In [9]:
top = purchases.nlargest(5, "TotalPrice")
for _, r in top.iterrows():
    ret = returns[(returns["Customer ID"] == r["Customer ID"]) & (returns["StockCode"] == r["StockCode"])]
    print(f"cust {r['Customer ID']}  {r['StockCode']}  bought {r['Quantity']:>6} (£{r['TotalPrice']:,.0f})  ->  matching return rows: {len(ret)}, qty {ret['Quantity'].sum()}")

cust 16446  23843  bought  80995 (£168,470)  ->  matching return rows: 1, qty -80995
cust 12346  23166  bought  74215 (£77,184)  ->  matching return rows: 1, qty -74215
cust 15098  22502  bought     60 (£38,970)  ->  matching return rows: 1, qty -60
cust 15838  84347  bought   9360 (£15,818)  ->  matching return rows: 2, qty -18720
cust 12918  M  bought      1 (£10,954)  ->  matching return rows: 2, qty -2


**Suggestion:** leave outliers alone for now and judge them in step 5, once we see the target distribution.

## 6. Save

`python src/step2_clean.py` writes `data/purchases.csv` and `data/returns.csv`. Later steps read those.

## What we learned

- 805,549 purchase rows, 5,878 customers; 18,744 return rows from 2,572 customers.
- All negative quantities were cancellations.
- Two decisions remain open: non-product codes, and extreme orders.